![QuantConnect Logo](https://cdn.quantconnect.com/web/i/icon.png)
<hr>

In [2]:
# 1. Imports and environment check
from AlgorithmImports import *
import os, json, importlib
import pandas as pd

print("pandas", pd.__version__)
missing = [m for m in ("statsmodels", "arch", "scipy", "pyarrow", "matplotlib") if importlib.util.find_spec(m) is None]
print("missing packages:", missing or "none")          # if any are missing, stop: the protocol needs them

import vrp_stages as stages
from vrp_config import CONFIG as cfg
from vrp_qc_source import QuantBookSource, run_probe
from vrp_qc_adapter import build_dataset, load_vix3m
import vrp_qc_source

# A running kernel caches imported modules, so an edited file is ignored until the kernel restarts.
# Reload everything in dependency order so edits take effect every time this cell runs.
for _m in ["vrp_config", "vrp_data_layer", "vrp_signals", "vrp_selection", "vrp_execution", "vrp_metrics",
           "vrp_engine", "vrp_protocol", "vrp_stages", "vrp_qc_adapter", "vrp_qc_source"]:
    importlib.reload(importlib.import_module(_m))
import vrp_stages as stages
from vrp_config import CONFIG as cfg
from vrp_qc_source import QuantBookSource, run_probe
from vrp_qc_adapter import build_dataset, load_vix3m
print("vrp_qc_source loaded from:", vrp_qc_source.__file__)
print("new probe code present:", "def step(name, fn)" in open(vrp_qc_source.__file__).read())

print("existing qc_data/ from an earlier session:", os.path.exists("qc_data"))   # tells you whether fetch progress survived
qb = QuantBook()
src = QuantBookSource(qb)

In [3]:
run_probe(src)

In [5]:
src.calibrate_index_shift()
src.calibrate_universe_shift(src.spx_daily("2014-01-01", "2024-12-31")["close"])

In [ ]:
vix3m = load_vix3m(src, "2005-01-01", pd.Timestamp.today().normalize())
vix3m.tail(3)

In [ ]:
report = build_dataset(src, cfg, "qc_data", history_start="2005-01-01", vix3m=vix3m)
report

In [ ]:
END = pd.Timestamp("2026-10-30")
def trim(name, col):
    p = f"qc_data/{name}.parquet"
    df = pd.read_parquet(p)
    df[df[col] <= END].reset_index(drop=True).to_parquet(p)
for n in ("spx_daily", "vix", "vix3m", "options_2026"):
    trim(n, "date")
trim("settlement", "expiry")
rep = json.load(open("qc_data/qc_build_report.json"))
rep["truncated_to"] = str(END.date())            # the other report fields describe the pre-truncation build
json.dump(rep, open("qc_data/qc_build_report.json", "w"), indent=2)

In [ ]:
ctx = stages.build_context(cfg, "qc_data", "qc_results", "qc_results/trial_log.csv")
R = stages.stage_insample(ctx, cfg)
text = open("qc_results/insample_report.md").read()
print(len(text), "characters")
print(text)

In [ ]:
R_is = json.load(open("qc_results/insample.json"))
R_ho, kill, verdict = stages.stage_holdout(ctx, cfg, R_is)
print("Verdict:", verdict)
for c in kill: print(f"[{c['status']:11}] {c['id']}. {c['name']}: {c['detail']}")